# PI Controller Optimiser — PLECS

Automatically tunes **Kp** and **Ki** by running closed-loop step-response
simulations in PLECS and minimising a weighted cost function.

**Workflow**
1. Edit **Cell 1 (Configuration)** for your model.
2. Run cells top-to-bottom.
3. The optimiser goes: *Grid search → Differential Evolution → Nelder-Mead*.
   You can skip any stage by setting the corresponding flag to `False`.

**Files required (same folder)**
- `plecs_sim_library.py`
- `signal_analysis_lib.py`
- `pi_optimizer_lib.py`

---
## Cell 1 — Configuration  *(only this cell needs editing)*

In [4]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                      CONFIGURATION CELL                             ║
# ║           Edit everything in this cell, leave the rest alone.       ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── PLECS connection ────────────────────────────────────────────────────────
PLECS_HOST   = "http://localhost:1080/RPC2"   # default PLECS XML-RPC port

# ── Model ───────────────────────────────────────────────────────────────────
MODEL_FOLDER = r"C:\Users\eliott.sefarang\Documents\Plexim\Test"   # absolute path, no trailing slash
MODEL_NAME   = "buck_converter_with_cascaded_controls"                     # without .plecs extension

# ── Variables always sent to PLECS (Mode A: full override)
# Leave empty {} to let PLECS use its own defaults (Mode B)
BASE_VARS = {
    # 'R'    : 0.5,
    # 'L'    : 3e-3,
    # 't_end': 0.5,
}

# ── PLECS variable names for the PI gains ───────────────────────────────────
# These must match the variable names used in your PLECS model's
# initialisation commands (e.g.  Kp = 1.0; Ki = 100;)
KP_VAR_NAME  = "kp"   # proportional gain variable name in PLECS
KI_VAR_NAME  = "ki"   # integral gain variable name in PLECS

# ── Signal indices (from the top-level Outport / Mux) ───────────────────────
# Run the 'Inspect Signals' cell (below) once to find the correct indices.
SIGNAL_IDX_MEASURED = 1    # row index of the measured / feedback signal
SIGNAL_IDX_REF      = 0    # row index of the reference / setpoint signal
                            # set to None if the setpoint is not in the Outport

# ── Step description ─────────────────────────────────────────────────────────
# The optimiser evaluates the response to a step from INITIAL_VALUE to SETPOINT.
SETPOINT      = 11    # target value after the step (same unit as signal)
INITIAL_VALUE = 7     # value before the step
T_STEP        = 0.025    # time at which the step occurs (seconds)

# ── Cost function weights ────────────────────────────────────────────────────
# Only the relative values matter — they are normalised automatically.
# Set a weight to 0 to exclude that metric from the optimisation.
#
#   Recommended starting point for a current/voltage control loop:
#     - Prioritise settling time and overshoot
#     - Use ITAE for a smooth, no-overshoot response
#     - Use ISE for a fast, possibly slightly overshooting response
COST_WEIGHTS = {
    "rise_time"     : 1.0,   # penalise slow rise
    "overshoot_pct" : 2.0,   # penalise overshoot (higher = stricter)
    "settling_time" : 3.0,   # penalise slow settling
    "sse"           : 2.0,   # penalise steady-state error
    "ise"           : 0.5,   # integral of squared error
    "itae"          : 0.5,   # integral of time × |error|
}

# ── Normalisation constants ──────────────────────────────────────────────────
# Each metric is divided by its normalisation constant before weighting.
# Tune these to match the expected scale of your system.
COST_NORMALISATION = {
    "rise_time"     : 1e-3,    # 5 ms is considered 'average'
    "overshoot_pct" : 5.0,    # 10 % is considered 'average'
    "settling_time" : 2e-3,   # 20 ms is considered 'average'
    "sse"           : 0.01,     # 0.1 (signal units) is considered 'average'
    "ise"           : 1e-3,
    "itae"          : 1e-4,
}

# ── Settling band (±% of step amplitude) ────────────────────────────────────
SETTLING_BAND = 0.02    # 0.02 = ±2 % of step amplitude

# ── Search bounds ───────────────────────────────────────────────────────────
# Define the range to explore for each gain.
# Use LOG_SCALE = True (recommended) for gains that may span several decades.
KP_BOUNDS  = (0.01,  50.0)   # (min, max) for Kp
KI_BOUNDS  = (1.0,  1000.0)  # (min, max) for Ki
LOG_SCALE  = True              # search in log10 space

# ── Stage 1: Grid search ────────────────────────────────────────────────────
# Optional but strongly recommended for first use: maps the cost landscape
# and reveals good starting bounds for the global optimiser.
RUN_GRID_SEARCH = False
N_KP_GRID       = 6     # number of Kp grid points
N_KI_GRID       = 6     # number of Ki grid points  (total = N_KP × N_KI sims)

# ── Stage 2: Differential Evolution ─────────────────────────────────────────
# Global optimiser — robust but slow. Reduce max_iter for faster runs.
RUN_DE       = True
DE_MAX_ITER  = 40     # maximum DE generations
DE_POPSIZE   = 10      # population size multiplier (total population = 10 × 2)
DE_TOL       = 1e-3   # convergence tolerance
DE_SEED      = 42     # fixed seed for reproducibility

# ── Stage 3: Nelder-Mead local refinement ───────────────────────────────────
# Polishes the DE result. Fast and precise near the minimum.
RUN_NM       = True
NM_MAX_ITER  = 50
NM_XATOL     = 1   # stop when gain change < this
NM_FATOL     = 1e-1   # stop when cost change < this

# ── Plot settings ───────────────────────────────────────────────────────────
# Time window for the best-response plot (seconds). None = full simulation.
PLOT_TIME_WINDOW = None        # e.g. (0.04, 0.20) to zoom around the step
SIGNAL_NAME_MEAS = "Measured" # label for the measured signal trace
SIGNAL_NAME_REF  = "Setpoint" # label for the reference trace

print("Configuration loaded.")

Configuration loaded.


---
## Cell 2 — Imports

In [5]:
import numpy as np

# PLECS runner — connection, loading, sweeping
from src.plecs_sim_library import (
    connect_plecs,
    load_model,
    close_model,
    inspect_signals,
    run_sweep,
)

# PI optimiser library
from src.pi_optimizer_lib import (
    compute_step_response_metrics,
    build_cost_function,
    evaluate_pi,
    grid_search_pi,
    optimize_pi_differential_evolution,
    optimize_pi_nelder_mead,
    plot_cost_landscape,
    plot_convergence,
    plot_best_response,
    print_optimization_summary,
    OptimHistory,
)

print("Imports OK.")

Imports OK.


---
## Cell 3 — Connect to PLECS and load model

In [6]:
server = connect_plecs(PLECS_HOST)
load_model(server, MODEL_FOLDER, MODEL_NAME)

[PLECS] Connected to http://localhost:1080/RPC2
[PLECS] Model 'buck_converter_with_cascaded_controls' loaded from 'C:\Users\eliott.sefarang\Documents\Plexim\Test'


---
## Cell 4 — Inspect signals  *(run once to find the correct signal indices)*

This runs one simulation with the current PLECS default gains and prints
every output signal with its index, min, max and mean.  
Use those indices to set `SIGNAL_IDX_MEASURED` and `SIGNAL_IDX_REF` in Cell 1.

In [7]:
# Run one simulation with base_vars only (no gain override) to see what comes out
init_results = run_sweep(
    server       = server,
    model_name   = MODEL_NAME,
    base_vars    = BASE_VARS,
    sweep_params = {KP_VAR_NAME: [BASE_VARS.get(KP_VAR_NAME, 1)], #1.0
                   KI_VAR_NAME: [BASE_VARS.get(KI_VAR_NAME, 100.0)]},#100.0
)
inspect_signals(init_results)

[Sweep] 1 step(s) | swept: ['kp', 'ki'] | mode: PLECS vars only (base_vars empty)
  step   1/1  ->  {'kp': 1, 'ki': 100.0}
[Sweep] Done.
Signals available in results (first run):
  Total signals  : 2  -> use signal_index = 0 to 1
  Total samples  : 114493
  Time range     : 0 s  to  0.1 s

  index                min             max            mean
  -------    -------------   -------------   -------------
  0                      7              11          10.321
  1                      0          11.003          10.155


---
## Cell 5 — Build the cost function

In [8]:
cost_fn = build_cost_function(
    weights       = COST_WEIGHTS,
    normalisation = COST_NORMALISATION,
)

# Quick sanity check on the initial default gains (from the inspect run)
init_res = init_results[0]
metrics_init = compute_step_response_metrics(
    time          = init_res['time'],
    measured      = init_res['values'][SIGNAL_IDX_MEASURED],
    setpoint      = SETPOINT,
    initial_value = INITIAL_VALUE,
    t_step        = T_STEP,
    settling_band = SETTLING_BAND,
)
cost_init = cost_fn(metrics_init)

print("Initial gains (PLECS defaults):")
print(f"  Rise time     = {metrics_init.rise_time*1e3:.2f} ms")
print(f"  Overshoot     = {metrics_init.overshoot_pct:.2f} %")
print(f"  Settling time = {metrics_init.settling_time*1e3:.2f} ms")
print(f"  SSE           = {metrics_init.sse:.4g}")
print(f"  ISE           = {metrics_init.ise:.4e}")
print(f"  ITAE          = {metrics_init.itae:.4e}")
print(f"  Cost          = {cost_init:.4f}  <- target to beat")

Initial gains (PLECS defaults):
  Rise time     = 8.39 ms
  Overshoot     = 0.06 %
  Settling time = 28.86 ms
  SSE           = 0.001942
  ISE           = 5.3648e-03
  ITAE          = 3.7384e-04
  Cost          = 6.2944  <- target to beat


---
## Cell 6 — Stage 1: Grid search  *(cost landscape)*

Maps the cost function over the full (Kp, Ki) space.  
Skip this cell (set `RUN_GRID_SEARCH = False` in Cell 1) on subsequent runs
once you are confident the DE bounds are well chosen.

In [9]:
history = OptimHistory()   # shared history across all stages

if RUN_GRID_SEARCH:
    KP_grid, KI_grid, COST_grid, history = grid_search_pi(
        server          = server,
        model_name      = MODEL_NAME,
        base_vars       = BASE_VARS,
        kp_range        = KP_BOUNDS,
        ki_range        = KI_BOUNDS,
        n_kp            = N_KP_GRID,
        n_ki            = N_KI_GRID,
        kp_var_name     = KP_VAR_NAME,
        ki_var_name     = KI_VAR_NAME,
        setpoint        = SETPOINT,
        initial_value   = INITIAL_VALUE,
        t_step          = T_STEP,
        signal_idx_meas = SIGNAL_IDX_MEASURED,
        cost_fn         = cost_fn,
        log_scale       = LOG_SCALE,
        settling_band   = SETTLING_BAND,
        history         = history,
    )

    plot_cost_landscape(
        KP        = KP_grid,
        KI        = KI_grid,
        COST      = COST_grid,
        history   = history,
        log_scale = LOG_SCALE,
        title     = "Cost landscape — grid search",
    )
else:
    print("Grid search skipped (RUN_GRID_SEARCH = False).")

Grid search skipped (RUN_GRID_SEARCH = False).


---
## Cell 7 — Stage 2: Differential Evolution  *(global optimiser)*

Searches the full (Kp, Ki) space without needing a starting point.  
This is the main optimisation step — it will run `DE_MAX_ITER × popsize × 2`
simulations in the worst case.

In [10]:
if RUN_DE:
    de_kp, de_ki, history = optimize_pi_differential_evolution(
        server          = server,
        model_name      = MODEL_NAME,
        base_vars       = BASE_VARS,
        kp_bounds       = KP_BOUNDS,
        ki_bounds       = KI_BOUNDS,
        kp_var_name     = KP_VAR_NAME,
        ki_var_name     = KI_VAR_NAME,
        setpoint        = SETPOINT,
        initial_value   = INITIAL_VALUE,
        t_step          = T_STEP,
        signal_idx_meas = SIGNAL_IDX_MEASURED,
        cost_fn         = cost_fn,
        max_iter        = DE_MAX_ITER,
        popsize         = DE_POPSIZE,
        tol             = DE_TOL,
        seed            = DE_SEED,
        log_scale       = LOG_SCALE,
        settling_band   = SETTLING_BAND,
        history         = history,
    )
    print(f"\nDE result:  Kp = {de_kp:.4g}   Ki = {de_ki:.4g}")
else:
    # Fall back to the best point found in the grid search
    de_kp, de_ki, _ = history.best()
    print(f"DE skipped. Using grid best:  Kp = {de_kp:.4g}   Ki = {de_ki:.4g}")

[DE] Starting  bounds Kp=(0.01, 50.0)  Ki=(1.0, 1000.0)  max_iter=40  popsize=10  seed=42
  [DE #  1]  Kp=7.702  Ki=2.106  cost=18.5341
  [DE #  2]  Kp=9.359  Ki=1.389  cost=17.9514
  [DE #  3]  Kp=0.07095  Ki=7.861  cost=63.0457
  [DE #  4]  Kp=0.5552  Ki=127.9  cost=3.8015
  [DE #  5]  Kp=0.3434  Ki=518.4  cost=0.7221
  [DE #  6]  Kp=43.71  Ki=824.2  cost=1.1036
  [DE #  7]  Kp=0.9176  Ki=33.18  cost=22.1517
  [DE #  8]  Kp=0.08483  Ki=348.6  cost=0.8937
  [DE #  9]  Kp=24.29  Ki=5.084  cost=5.0430
  [DE # 10]  Kp=2.012  Ki=82.75  cost=11.5212
  [DE # 11]  Kp=21.02  Ki=19  cost=15.5293
  [DE # 12]  Kp=0.02091  Ki=8.548  cost=54.7131
  [DE # 13]  Kp=0.03678  Ki=11.95  cost=36.6420
  [DE # 14]  Kp=0.01173  Ki=188.6  cost=1.5299
  [DE # 15]  Kp=2.762  Ki=3.801  cost=36.3379
  [DE # 16]  Kp=0.02505  Ki=106.4  cost=2.9074
  [DE # 17]  Kp=4.999  Ki=24.76  cost=18.9187
  [DE # 18]  Kp=0.2129  Ki=1.737  cost=149.5860
  [DE # 19]  Kp=1.226  Ki=50.69  cost=17.0437
  [DE # 20]  Kp=0.1835  Ki=46

---
## Cell 8 — Stage 3: Nelder-Mead local refinement

Polishes the DE result with a gradient-free simplex method.
Typically converges in < 30 simulations.

In [11]:
if RUN_NM:
    best_kp, best_ki, history = optimize_pi_nelder_mead(
        server          = server,
        model_name      = MODEL_NAME,
        base_vars       = BASE_VARS,
        kp_start        = de_kp,
        ki_start        = de_ki,
        kp_var_name     = KP_VAR_NAME,
        ki_var_name     = KI_VAR_NAME,
        setpoint        = SETPOINT,
        initial_value   = INITIAL_VALUE,
        t_step          = T_STEP,
        signal_idx_meas = SIGNAL_IDX_MEASURED,
        cost_fn         = cost_fn,
        max_iter        = NM_MAX_ITER,
        xatol           = NM_XATOL,
        fatol           = NM_FATOL,
        log_scale       = LOG_SCALE,
        settling_band   = SETTLING_BAND,
        history         = history,
    )
    print(f"\nFinal result:  Kp = {best_kp:.6g}   Ki = {best_ki:.6g}")
else:
    best_kp, best_ki = de_kp, de_ki
    print(f"NM skipped. Using:  Kp = {best_kp:.6g}   Ki = {best_ki:.6g}")

[NM] Starting from Kp=0.4577  Ki=995.2  max_iter=50
  [NM #  1]  Kp=0.4577  Ki=995.2  cost=0.3236
  [NM #  2]  Kp=0.6465  Ki=995.2  cost=0.3839
  [NM #  3]  Kp=0.4577  Ki=1406  cost=0.5046
  [NM #  4]  Kp=0.6465  Ki=704.5  cost=0.6221
  [NM #  5]  Kp=0.4989  Ki=1183  cost=0.2987

[NM] Done  ->  Kp=0.498925  Ki=1182.77  cost=0.2987  (converged, 5 evals)

Final result:  Kp = 0.498925   Ki = 1182.77


---
## Cell 9 — Convergence plot

In [12]:
plot_convergence(
    history   = history,
    title     = "Optimisation convergence  (GRID → DE → NM)",
    show_best = True,
)

---
## Cell 10 — Best response plot

Re-runs PLECS with the optimised gains and renders an annotated step-response
chart with rise time, settling time, overshoot and settling band.

In [13]:
best_metrics = plot_best_response(
    server           = server,
    model_name       = MODEL_NAME,
    base_vars        = BASE_VARS,
    best_kp          = best_kp,
    best_ki          = best_ki,
    kp_var_name      = KP_VAR_NAME,
    ki_var_name      = KI_VAR_NAME,
    setpoint         = SETPOINT,
    initial_value    = INITIAL_VALUE,
    t_step           = T_STEP,
    signal_idx_meas  = SIGNAL_IDX_MEASURED,
    cost_fn          = cost_fn,
    signal_idx_ref   = SIGNAL_IDX_REF,
    signal_name_meas = SIGNAL_NAME_MEAS,
    signal_name_ref  = SIGNAL_NAME_REF,
    time_window      = PLOT_TIME_WINDOW,
    settling_band    = SETTLING_BAND,
    title            = "Optimised PI — Step response",
)

[Plot] Running best simulation  Kp=0.4989  Ki=1183 ...


---
## Cell 11 — Optimisation summary

In [14]:
best_cost = cost_fn(best_metrics)

print(f"Initial cost  : {cost_init:.4f}")
print(f"Optimised cost: {best_cost:.4f}")
print(f"Improvement   : {(1 - best_cost / cost_init) * 100:.1f} %\n")

print_optimization_summary(
    best_kp = best_kp,
    best_ki = best_ki,
    metrics = best_metrics,
    cost    = best_cost,
)

Initial cost  : 6.2944
Optimised cost: 0.2987
Improvement   : 95.3 %

──────────────────────────────────────────────────
  PI OPTIMISATION RESULT
──────────────────────────────────────────────────
  Kp              = 0.498925
  Ki              = 1182.77
──────────────────────────────────────────────────
  Rise time       = 0.39 ms
  Overshoot       = 0.66 %
  Settling time   = 0.65 ms
  Steady-state Δ  = 5.45709e-05  (abs)
  ISE             = 1.7544e-03
  ITAE            = 3.3613e-05
──────────────────────────────────────────────────
  Cost (weighted) = 0.298703
──────────────────────────────────────────────────


---
## Cell 12 — (Optional) Manual verification sweep

Sweep a few Kp values around the optimum to visually confirm the minimum
and check robustness to gain variation.

In [15]:
from src.plecs_sim_library import plot_signals

# Explore ±50 % around the optimised Kp, keeping Ki fixed
kp_sweep_vals = [best_kp * f for f in [0.5, 0.75, 1.0, 1.25, 1.5]]

verification_results = run_sweep(
    server       = server,
    model_name   = MODEL_NAME,
    base_vars    = {**BASE_VARS, KI_VAR_NAME: best_ki},
    sweep_params = {KP_VAR_NAME: kp_sweep_vals},
)

plot_signals(
    results     = verification_results,
    plot_config = [
        {
            'signal_index': SIGNAL_IDX_MEASURED,
            'signal_name' : SIGNAL_NAME_MEAS,
            'time_window' : PLOT_TIME_WINDOW,
            'title'       : f'Kp sensitivity sweep (Ki = {best_ki:.4g})',
            'ylabel'      : 'Amplitude',
            'show_legend' : True,
        },
    ],
    sim_mode    = 'sweep',
    label_param = KP_VAR_NAME,
    max_points  = 50_000,
)

[Sweep] 5 step(s) | swept: ['kp'] | mode: Python + PLECS vars
  base_vars sent every run: ['ki']


TypeError: cannot marshal <class 'numpy.float64'> objects

---
## Cell 13 — Close PLECS model

In [ ]:
close_model(server, MODEL_NAME)